# Điểm kiểm đối chứng kích thước tập (size- và class-matched)

Notebook này **KHÔNG** tạo điểm kiểm point-in-time. Nó rút đúng số nhãn mỗi lớp mà điểm kiểm point-in-time đã dùng (167 NEGATIVE / 616 NEUTRAL / 266 POSITIVE), nhưng lấy từ mọi mốc thời gian.

Đọc ba cột cạnh nhau — đầy đủ 1.306 / point-in-time 1.049 / đối chứng 1.049 — cho biết đóng góp cảm xúc thay đổi vì loại bỏ nhãn tương lai hay đơn giản vì tập huấn luyện nhỏ hơn.

**Số seed quyết định sức mạnh của kết luận.** Tập con nào được rút cũng là một yếu tố ngẫu nhiên, nên **một seed duy nhất chỉ là thăm dò**: không phân biệt được ảnh hưởng của kích thước tập với ảnh hưởng của một lần rút không may. Chạy `SAMPLE_SEED` lần lượt 7, 8, 9 (mỗi lần đổi cả `OUTPUT_DIR`) để có một dải so được với khoảng tin cậy của đóng góp thông tin. Nếu chỉ chạy được một seed, báo cáo phải gọi nó là thăm dò.

Trên Kaggle: bật **GPU** và **Internet**, đính kèm Dataset `phuocthoai/stock-trend-forecasting` chứa đúng một tệp `labeled_merged.csv`, và đính kèm ZIP điểm kiểm point-in-time (notebook tự tìm `manifest.json` cả trong ZIP). Chạy từ trên xuống.

In [ ]:
from pathlib import Path
import json
import importlib
import importlib.util
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nphuoctho/stock-trend-forecasting.git'
BRANCH = 'feat/point-in-time-checkpoint'
REPO_DIR = Path('/kaggle/working/stock-trend-forecasting') if Path('/kaggle').exists() else Path('/content/stock-trend-forecasting')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', '--detach', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
SUBPROCESS_ENV = {**os.environ, 'PYTHONPATH': os.pathsep.join(part for part in (str(SRC_DIR), os.environ.get('PYTHONPATH', '')) if part)}

# Giữ torch CUDA của Kaggle. Cặp này đã được ghi trong artifact CV thành công.
if importlib.util.find_spec('torchvision') is not None:
    check = subprocess.run([sys.executable, '-c', 'import torchvision'], capture_output=True, text=True)
    if check.returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'timm'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers==5.15.1', 'tokenizers>=0.22,<=0.23.0', 'huggingface-hub>=1.5,<2.0', 'accelerate>=1.1,<2', 'sentencepiece>=0.2,<1'], check=True)
importlib.invalidate_caches()
subprocess.run([sys.executable, '-c', "import torch; assert torch.__version__ == '2.10.0+cu128', f'PyTorch {torch.__version__}; cần phiên Kaggle mới với 2.10.0+cu128.'"], check=True, env=SUBPROCESS_ENV)
# Probe import Trainer trong subprocess: nếu transformers 5.x SIGSEGV (đã gặp trên
# Kaggle với cặp torch pinned này), returncode != 0 chứ kernel không chết.
probe = subprocess.run([sys.executable, '-c', "from transformers import Trainer, TrainingArguments; TrainingArguments(output_dir='/tmp/stf-check', eval_strategy='no', save_strategy='no', use_cpu=True)"], env=SUBPROCESS_ENV)
if probe.returncode != 0:
    # Fallback sang stack đã chạy ổn định trong repo (transformers 4.46.3).
    # eval_strategy/save_only_model đều có từ 4.41/4.38 nên code repo tương thích.
    print('transformers 5.x probe failed (rc=%s); falling back to 4.46.3' % probe.returncode)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'transformers==4.46.3', 'tokenizers>=0.20,<0.21',
                    'huggingface-hub>=0.23,<1.0', 'accelerate>=0.26,<1',
                    'sentencepiece>=0.2,<1'], check=True)
    importlib.invalidate_caches()
    subprocess.run([sys.executable, '-c', "from transformers import Trainer, TrainingArguments; TrainingArguments(output_dir='/tmp/stf-check', eval_strategy='no', save_strategy='no', use_cpu=True)"], check=True, env=SUBPROCESS_ENV)
    print('Fallback stack OK: transformers 4.46.3')
print('Repository ready:', REPO_DIR)

In [ ]:
import torch
import pandas as pd
from stf.sentiment.dataset import file_fingerprint, load_labeled

if not torch.cuda.is_available():
    raise RuntimeError('GPU chưa được bật. Chọn T4 GPU rồi chạy lại notebook.')
INPUT_ROOT = Path('/kaggle/input')
matches = sorted(path for path in INPUT_ROOT.rglob('labeled_merged.csv') if path.is_file())
if len(matches) != 1:
    found = '\n'.join(str(path) for path in matches) or '(không có tệp nào)'
    raise FileNotFoundError(
        'Cần đúng một tệp labeled_merged.csv trong Kaggle Input. Tìm thấy:\n' + found
    )
DATA_PATH = matches[0]
print('Using attached label file:', DATA_PATH)
raw = pd.read_csv(DATA_PATH)
assert len(raw) == 1306, f'Cần 1.306 nhãn đã hợp nhất, nhận {len(raw)}.'
assert raw['annotation_source'].eq('human_reviewed').all()
assert raw['annotation_status'].eq('REVIEWED').all()
reviewed = load_labeled(DATA_PATH)
print('GPU:', torch.cuda.get_device_name(0))
print('Rows:', len(reviewed))
print('SHA-256:', file_fingerprint(DATA_PATH))
print(reviewed['label'].value_counts().to_string())

In [ ]:
# Locate the point-in-time manifest among the attached Kaggle inputs. Kaggle does
# not unpack an uploaded ZIP, so search loose files AND zip members. `rglob` alone
# would also happily pick up an unrelated manifest.json, so require exactly one
# whose provenance matches the reference this control is supposed to mirror.
import zipfile

EXPECTED_CUTOFF = '2024-10-21'
EXPECTED_SIZE = 1049


def _matches(blob):
    return (
        isinstance(blob, dict)
        and blob.get('run_type') == 'full_data_refit'
        and blob.get('provenance', {}).get('label_cutoff') == EXPECTED_CUTOFF
        and blob.get('training_size') == EXPECTED_SIZE
        and blob.get('class_distribution')
    )


candidates = []
seen = []  # every manifest found, matched or not, so a mismatch is diagnosable


def _consider(blob, location, extract_to=None):
    if not isinstance(blob, dict):
        seen.append(f'{location} | (không phải manifest)')
        return
    prov = blob.get('provenance', {})
    seen.append(
        f'{location} | run_type={blob.get("run_type")} '
        f'label_cutoff={prov.get("label_cutoff")} '
        f'training_size={blob.get("training_size")}'
    )
    if _matches(blob):
        candidates.append((extract_to or location, blob))


for path in Path('/kaggle/input').rglob('manifest.json'):
    try:
        blob = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        continue
    _consider(blob, path)

for archive in Path('/kaggle/input').rglob('*.zip'):
    try:
        with zipfile.ZipFile(archive) as zf:
            for member in zf.namelist():
                if not member.endswith('manifest.json'):
                    continue
                try:
                    blob = json.loads(zf.read(member).decode('utf-8'))
                except (KeyError, ValueError):
                    continue
                if _matches(blob):
                    # Extract so the CLI can read it by path.
                    zf.extract(member, '/kaggle/working/reference')
                _consider(blob, f'{archive}!{member}',
                          Path('/kaggle/working/reference') / member)
    except (OSError, zipfile.BadZipFile):
        continue

if len(candidates) != 1:
    report = '\n'.join(seen) or '(không tìm thấy manifest.json nào trong Input)'
    raise FileNotFoundError(
        f'Cần đúng một manifest point-in-time với label_cutoff={EXPECTED_CUTOFF} '
        f'và training_size={EXPECTED_SIZE}. Hãy đính kèm output của notebook '
        f'point-in-time (Add Input -> Your Work) hoặc tải ZIP của nó lên làm '
        f'Dataset. Các manifest đã thấy:\n{report}'
    )
PIT_MANIFEST, PIT_BLOB = candidates[0]
print('Reference manifest:', PIT_MANIFEST)
print('Class counts to match:', PIT_BLOB['class_distribution'])

# Đổi seed này (7, 8, 9) cho mỗi lượt đối chứng; OUTPUT_DIR đổi theo.
SAMPLE_SEED = 7

INPUT_VARIANT = 'title_context'
TRUNCATION_STRATEGY = 'head_tail'
CLASS_WEIGHTING = 'inverse_frequency'
EPOCHS = 5
BATCH_SIZE = 16
SEED = 42
CV_RESULTS = REPO_DIR / 'outputs/sentiment-cv-merged/cv_results.json'
OUTPUT_DIR = (
    Path(f'/kaggle/working/size-matched-s{SAMPLE_SEED}')
    if Path('/kaggle').exists()
    else Path(f'/content/size-matched-s{SAMPLE_SEED}')
)

command = [
    sys.executable, '-m', 'stf.cli', 'sentiment-refit',
    '--data', str(DATA_PATH),
    '--cv-results', str(CV_RESULTS),
    '--input-variant', INPUT_VARIANT,
    '--truncation-strategy', TRUNCATION_STRATEGY,
    '--class-weighting', CLASS_WEIGHTING,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--seed', str(SEED),
    '--sample-like', str(PIT_MANIFEST),
    '--sample-seed', str(SAMPLE_SEED),
    '--output', str(OUTPUT_DIR),
]
print(' '.join(command))
subprocess.run(command, check=True, env=SUBPROCESS_ENV)

In [ ]:
import json

manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['run_type'] == 'full_data_refit'
assert manifest['training_size'] == 1049
assert manifest['provenance']['source_file_sha256'] == file_fingerprint(DATA_PATH)
assert 'test_metrics' not in manifest
subset = manifest['provenance']['subset']
assert subset['strategy'] == 'class_matched_random'
assert subset['point_in_time'] is False
assert subset['sample_seed'] == SAMPLE_SEED
assert manifest['class_distribution'] == PIT_BLOB['class_distribution']
print('Subset id:', subset['selected_sample_id_sha256'][:16], '| seed', subset['sample_seed'])
assert (OUTPUT_DIR / 'best' / 'config.json').is_file()
print(json.dumps(manifest['selection'], ensure_ascii=False, indent=2))
print('Checkpoint:', OUTPUT_DIR / 'best')

In [ ]:
import shutil

archive_base = Path.cwd() / OUTPUT_DIR.name
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
print('Archive:', archive_path)
print(f'Size: {archive_path.stat().st_size / 1024**2:.1f} MiB')
print('Kaggle: chọn Save Version → Save & Run All, rồi tải ZIP ở tab Output của phiên bản đã lưu.')